# graph

> Graph-spine EXTENSION + skeptical-lens verification (stage 5, CR-18 revolution 2).
> Decomp no longer creates a Document: it RECOMPUTES the transcription-emitted root's
> deterministic node ids from the consumed manifest (no search), verifies the root
> exists, and attaches the fine `Segment` spine under the existing `AudioSegment`
> nodes — PART_OF to the owning AudioSegment, STARTS_WITH per AudioSegment (the
> coarse-seam jump anchor), source-wide NEXT. Each Segment carries the audio
> `TimeSlice` ref plus per-transcriber `CharSlice` refs into the Transcript nodes
> (the D4/P10 framing, finally expressible). Commit goes through the layer's
> idempotent `extend_graph`.

In [ ]:
#| default_exp graph

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

from cjm_substrate.core.queue import JobQueue
from cjm_context_graph_primitives.provenance import SourceRef
from cjm_context_graph_primitives.graph import GraphNode
# Stage 4: typed query expressions + results (importing the result classes IS
# the host-side wire registration, F8)
from cjm_context_graph_primitives.query import (
    NodeQuery, EdgeQuery, PropertyPredicate, RelationPredicate, OrderBy,
    NodeQueryResult, EdgeQueryResult,
)
# Stage 5: the graph-aware layer owns the shared plumbing (graph_task is THE
# shared copy now; extend_graph = emit-if-absent + verify-if-present) and the
# fractal spine grammar; the audio-transcript schema owns the node nouns +
# deterministic identity tuples. cjm_graph_domains is GONE (Document dissolved).
from cjm_context_graph_layer.ops import GRAPH_TASK, graph_task, extend_graph
from cjm_context_graph_layer.grammar import SpineRelations, grouped_spine_edges
from cjm_transcript_graph_schema.schema import (
    TranscriptGraphLabels, SegmentNode, TranscriptSliceRef,
    source_node_id, audio_segment_node_id, audio_rendition_node_id, transcript_node_id,
)

from cjm_transcript_decomp_core.models import DecompSegment

In [ ]:
#| export
def resolve_root_ids(
    source_entry: Dict[str, Any],            # One source entry from the transcription manifest (0.3.0)
    capabilities_info: Dict[str, Dict[str, Any]], # The transcription manifest's capabilities block (config hashes)
) -> Dict[str, Any]:  # {"source", "chain", "audio_segments": [{audio_segment, rendition, start, end, model_input_hash, transcripts}]}
    """Recompute the transcription-emitted root node ids from manifest data.

    Deterministic identity makes the extender lookup a RECOMPUTATION, not a
    search: Source = content hash; AudioSegment = (source, boundary range);
    AudioRendition = (audio segment, preprocessing chain) — the per-source
    `chain` ([] = raw convert-only) carried by the 0.3.0 manifest; Transcript =
    (rendition, transcriber, config_hash). No stored-id coupling between the
    cores. A pre-0.3.0 manifest (no `chain`) recomputes the raw rendition ids
    (empty chain), which is correct for any non-preprocessed run.
    """
    content_hash = str(source_entry.get("content_hash") or "")
    if not content_hash:
        raise ValueError(
            f"source {source_entry.get('source_path')!r} has no content_hash — "
            "re-run transcription on the 0.3.0 manifest schema")
    source_id = source_node_id(content_hash)
    chain = list(source_entry.get("chain") or [])  # [] = raw convert-only rendition
    asegs: List[Dict[str, Any]] = []
    for pseg in source_entry.get("segments") or []:
        start, end = float(pseg.get("start", 0.0)), float(pseg.get("end", 0.0))
        aseg_id = audio_segment_node_id(source_id, start, end)
        rendition_id = audio_rendition_node_id(aseg_id, chain)
        transcripts = {
            t: transcript_node_id(rendition_id, t, str((capabilities_info.get(t) or {}).get("config_hash") or ""))
            for t in (pseg.get("transcripts") or {})
        }
        asegs.append({"audio_segment": aseg_id, "rendition": rendition_id,
                      "start": start, "end": end,
                      "model_input_hash": str(pseg.get("model_input_hash") or ""),
                      "transcripts": transcripts})
    return {"source": source_id, "chain": chain, "audio_segments": asegs}

In [ ]:
#| export
def build_extension_payload(
    source_entry: Dict[str, Any],             # One source entry from the transcription manifest
    capabilities_info: Dict[str, Dict[str, Any]],  # The transcription manifest's capabilities block
    vad_config_hash: str,                     # THIS run's VAD capability config hash (skeleton identity input)
    text_from: str,                           # Authoritative transcriber (layer-0 text designation)
    segments: List[DecompSegment],            # Ordered aligned segments (per-transcriber variants attached)
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:  # (nodes, edges, ids)
    """Build the fine-spine EXTENSION payload (pure; no capability calls).

    Segment identity is audio-side (audio RENDITION, VAD config, chunk range) —
    shared across transcribers by construction, distinct per rendition (vocals
    isolation can yield different VAD chunking than raw). Each Segment carries
    the audio `TimeSlice` ref into its rendition + one `CharSlice` ref per
    transcriber variant; `text_from` is recorded per segment as provenance
    (which Transcript the layer-0 text came from), never as global config. Edges
    come from `grouped_spine_edges`: PART_OF to the OWNING AudioRendition,
    STARTS_WITH per rendition, NEXT chained source-wide across coarse boundaries.
    """
    roots = resolve_root_ids(source_entry, capabilities_info)
    source_id = roots["source"]
    asegs = roots["audio_segments"]

    nodes: List[Dict[str, Any]] = []
    seg_ids: List[str] = []
    groups: Dict[int, List[str]] = {}
    used_transcripts: set = set()
    for seg in segments:
        a = asegs[seg.pseg_index]
        slices: List[TranscriptSliceRef] = []
        for var in seg.variants:
            tid = a["transcripts"].get(var.transcriber)
            if tid is None or not var.text or var.start_char is None or var.end_char is None:
                continue
            slices.append(TranscriptSliceRef(transcript=tid, start_char=var.start_char,
                                             end_char=var.end_char, text=var.text))
            used_transcripts.add(tid)
        node = SegmentNode(
            rendition=a["rendition"], vad_config_hash=vad_config_hash,
            chunk_start=seg.chunk_start, chunk_end=seg.chunk_end,
            index=seg.index, start_time=seg.start_time, end_time=seg.end_time,
            text=seg.text, audio_hash=a["model_input_hash"], source=source_id,
            text_from=(a["transcripts"].get(text_from) if seg.text.strip() else None),
            text_slices=slices,
        )
        nodes.append(node.to_graph_node())
        seg_ids.append(node.id)
        groups.setdefault(seg.pseg_index, []).append(node.id)

    # The fine spine hangs under each segment's RENDITION (PART_OF rendition).
    edges = grouped_spine_edges([(asegs[i]["rendition"], groups[i]) for i in sorted(groups)])
    ids = {"source": source_id, "segments": seg_ids,
           "audio_segments": [a["audio_segment"] for a in asegs],
           "renditions": [a["rendition"] for a in asegs],
           "transcripts_used": sorted(used_transcripts)}
    return nodes, edges, ids

In [ ]:
#| export
@dataclass
class SourceVerification:
    """Skeptical-lens verification of one Source's fine-spine extension under a
    specific rendition set, computed by querying the graph directly (never
    trusting run state)."""
    source_id: str             # Verified Source node id
    title: str                 # Source title (read back from the graph)
    audio_segment_count: int   # AudioSegment nodes found under the Source (coarse spine; rendition-independent)
    rendition_count: int       # AudioRendition nodes the fine spine was scoped to (the run's extended renditions)
    segment_count: int         # Fine Segment nodes found under those renditions
    source_starts_with: bool   # Exactly 1 STARTS_WITH Source -> first AudioSegment
    aseg_next_complete: bool   # Coarse NEXT count == audio_segment_count - 1
    seg_next_complete: bool    # Fine NEXT count == segment_count - 1 (source-wide chain)
    part_of_complete: bool     # Fine PART_OF count == segment_count (Segment -> rendition)
    rendition_starts_with_complete: bool  # STARTS_WITH from renditions == #renditions owning >=1 Segment
    all_have_timing: bool      # Every Segment has start_time + end_time
    all_have_sources: bool     # Every Segment has >=1 SourceRef (the audio ref at minimum)
    source_locators: List[str] = field(default_factory=list)  # Distinct provenance locator URIs

    @property
    def ok(self) -> bool:  # True when every structural check passes
        """All structural checks pass."""
        return (self.source_starts_with and self.aseg_next_complete
                and self.seg_next_complete and self.part_of_complete
                and self.rendition_starts_with_complete and self.all_have_timing
                and self.all_have_sources)

In [ ]:
#| export
async def verify_source(
    queue: JobQueue,          # Started job queue
    graph_id: str,            # Graph-storage capability id
    source_id: str,           # Source node id to verify
    rendition_ids: List[str], # The AudioRendition ids the run extended (the fine spine is scoped to these)
) -> Optional[SourceVerification]:  # Result, or None if the Source is not found
    """Verify a Source's committed extension via server-side AGGREGATES (D13/D19).

    Two-layer verification for the Source-rooted schema: the coarse spine
    (Source → AudioSegments) is checked with id-list edge counts (rendition-
    independent); the fine spine hangs under the run's AudioRendition set, so it
    is scoped through the batched far-end constraint
    `RelationPredicate("PART_OF", node_ids=rendition_ids)` (the C17 batch shape) —
    never a whole-neighborhood materialization, and never mixing a sibling
    rendition's spine (raw vs vocals coexist under the same AudioSegments). One
    bounded projection pass (sources + rendition_id) yields missing-sources,
    distinct locators, AND the per-rendition ownership set for the STARTS_WITH
    check.
    """
    src = await graph_task(queue, graph_id, "get_node", node_id=source_id)
    if src is None:
        return None
    props = src.properties if isinstance(src, GraphNode) else ((src or {}).get("properties") or {})

    async def _ecount(**kw):
        q = EdgeQuery(count=True, **kw)
        res = await graph_task(queue, graph_id, "query_edges", query=q.to_dict())
        return int(res.count or 0)

    # Coarse layer: ordered AudioSegment ids under the Source (rendition-independent).
    aq = NodeQuery(label=TranscriptGraphLabels.AUDIO_SEGMENT,
                   related=RelationPredicate(SpineRelations.PART_OF, node_id=source_id),
                   order_by=OrderBy("index"), project=["index"])
    ares = await graph_task(queue, graph_id, "query_nodes", query=aq.to_dict())
    aseg_ids = [r["id"] for r in (ares.rows or [])]
    n_asegs = len(aseg_ids)
    rendition_ids = list(rendition_ids or [])
    if not aseg_ids or not rendition_ids:
        return SourceVerification(
            source_id=source_id, title=props.get("title", "Untitled"),
            audio_segment_count=n_asegs, rendition_count=len(rendition_ids),
            segment_count=0, source_starts_with=False,
            aseg_next_complete=False, seg_next_complete=False, part_of_complete=False,
            rendition_starts_with_complete=False, all_have_timing=False, all_have_sources=False)

    src_starts = await _ecount(relation_type=SpineRelations.STARTS_WITH, source_id=source_id)
    aseg_next = await _ecount(relation_type=SpineRelations.NEXT, source_ids=aseg_ids)

    # Fine layer, scoped via the batched far-end constraint onto THIS run's renditions.
    part_of_rend = RelationPredicate(SpineRelations.PART_OF, node_ids=rendition_ids)

    async def _ncount(**kw):
        q = NodeQuery(label=TranscriptGraphLabels.SEGMENT, related=part_of_rend, count=True, **kw)
        res = await graph_task(queue, graph_id, "query_nodes", query=q.to_dict())
        return int(res.count or 0)

    n_segs = await _ncount()
    seg_next = await _ecount(relation_type=SpineRelations.NEXT, source_related=part_of_rend)
    seg_part_of = await _ecount(relation_type=SpineRelations.PART_OF, target_ids=rendition_ids)
    rend_starts = await _ecount(relation_type=SpineRelations.STARTS_WITH, source_ids=rendition_ids)
    missing_start = await _ncount(where=[PropertyPredicate("start_time", "is_null")])
    missing_end = await _ncount(where=[PropertyPredicate("end_time", "is_null")])

    # Bounded projection: sources + owning rendition in ONE pass.
    sq = NodeQuery(label=TranscriptGraphLabels.SEGMENT, related=part_of_rend,
                   project=["sources", "rendition_id"])
    res = await graph_task(queue, graph_id, "query_nodes", query=sq.to_dict())
    missing_sources = 0
    locators = set()
    owning_renditions = set()
    for r in (res.rows or []):
        sources = r.get("sources") or []
        if not sources:
            missing_sources += 1
        for s in sources:
            ref = SourceRef.from_dict(s) if isinstance(s, dict) else s
            locators.add(ref.locator.to_uri())
        if r.get("rendition_id"):
            owning_renditions.add(r["rendition_id"])

    return SourceVerification(
        source_id=source_id,
        title=props.get("title", "Untitled"),
        audio_segment_count=n_asegs,
        rendition_count=len(rendition_ids),
        segment_count=n_segs,
        source_starts_with=src_starts == 1,
        aseg_next_complete=aseg_next == max(0, n_asegs - 1),
        seg_next_complete=seg_next == max(0, n_segs - 1),
        part_of_complete=seg_part_of == n_segs,
        rendition_starts_with_complete=rend_starts == len(owning_renditions),
        all_have_timing=(missing_start == 0 and missing_end == 0),
        all_have_sources=missing_sources == 0,
        source_locators=sorted(locators),
    )

In [ ]:
# extension payload smoke checks (pure; no capabilities)
from cjm_transcript_decomp_core.models import SegmentVariant

_source_entry = {
    "source_path": "/media/ep1.mp3",
    "content_hash": "sha256:src",
    "segments": [
        {"start": 0.0, "end": 300.0, "model_input_hash": "sha256:wav0",
         "transcripts": {"whisper": {}, "voxtral": {}}},
        {"start": 300.0, "end": 600.0, "model_input_hash": "sha256:wav1",
         "transcripts": {"whisper": {}, "voxtral": {}}},
    ],
}
_capabilities = {"whisper": {"config_hash": "sha256:cw"}, "voxtral": {"config_hash": "sha256:cv"}}

roots = resolve_root_ids(_source_entry, _capabilities)
assert roots["source"] == source_node_id("sha256:src") and roots["chain"] == []
_a0 = audio_segment_node_id(roots["source"], 0.0, 300.0)
assert roots["audio_segments"][0]["audio_segment"] == _a0
# raw rendition (empty chain) + transcript keyed on the RENDITION
_r0 = audio_rendition_node_id(_a0, [])
assert roots["audio_segments"][0]["rendition"] == _r0
assert roots["audio_segments"][1]["transcripts"]["voxtral"] == transcript_node_id(
    roots["audio_segments"][1]["rendition"], "voxtral", "sha256:cv")
# a preprocessed manifest (chain set) recomputes DISTINCT rendition/transcript ids,
# SAME audio-segment ids — raw + vocals fine spines coexist under one boundary.
_roots_vox = resolve_root_ids({**_source_entry, "chain": ["source_separation:demucs@cfg"]}, _capabilities)
assert _roots_vox["audio_segments"][0]["audio_segment"] == _a0
assert _roots_vox["audio_segments"][0]["rendition"] != _r0
assert _roots_vox["audio_segments"][0]["transcripts"]["voxtral"] != roots["audio_segments"][0]["transcripts"]["voxtral"]

# 3 fine segments: 2 under pseg 0, 1 under pseg 1 (NEXT must cross the boundary)
_segs = [
    DecompSegment(index=0, text="Alpha.", start_time=0.0, end_time=2.0,
                  chunk_start=0.0, chunk_end=2.0, vad_chunk_index=0, pseg_index=0,
                  variants=[SegmentVariant("voxtral", "Alpha.", 0, 6),
                            SegmentVariant("whisper", "alfa", 0, 4)]),
    DecompSegment(index=1, text="Beta.", start_time=2.0, end_time=4.0,
                  chunk_start=2.0, chunk_end=4.0, vad_chunk_index=1, pseg_index=0,
                  variants=[SegmentVariant("voxtral", "Beta.", 7, 12)]),
    DecompSegment(index=2, text="", start_time=300.5, end_time=301.0,
                  chunk_start=0.5, chunk_end=1.0, vad_chunk_index=0, pseg_index=1,
                  variants=[]),  # D14-class empty segment: audio ref only
]
nodes, edges, ids = build_extension_payload(_source_entry, _capabilities, "sha256:vad", "voxtral", _segs)
assert len(nodes) == 3 and ids["source"] == roots["source"]
assert ids["renditions"] == [a["rendition"] for a in roots["audio_segments"]]
# deterministic + audio-side identity: rebuild reproduces ids byte-identically
nodes2, edges2, ids2 = build_extension_payload(_source_entry, _capabilities, "sha256:vad", "voxtral", _segs)
assert ids2["segments"] == ids["segments"] and [e["id"] for e in edges2] == [e["id"] for e in edges]

# edge topology: STARTS_WITH per OWNING rendition (2), PART_OF per segment (3),
# NEXT source-wide (2 — crosses the pseg boundary); PART_OF targets the rendition
_rels = [e["relation_type"] for e in edges]
assert _rels.count("STARTS_WITH") == 2 and _rels.count("PART_OF") == 3 and _rels.count("NEXT") == 2
_part_of_targets = {e["target_id"] for e in edges if e["relation_type"] == "PART_OF"}
assert _part_of_targets <= set(ids["renditions"]), "fine spine PART_OF the rendition"
_nexts = [(e["source_id"], e["target_id"]) for e in edges if e["relation_type"] == "NEXT"]
assert _nexts == [(ids["segments"][0], ids["segments"][1]), (ids["segments"][1], ids["segments"][2])]

# provenance: audio TimeSlice ref + per-transcriber CharSlice refs; text_from recorded
n0 = nodes[0]
assert n0["properties"]["text_from"] == roots["audio_segments"][0]["transcripts"]["voxtral"]
assert n0["properties"]["rendition_id"] == _r0
assert len(n0["sources"]) == 3  # audio + 2 variants
assert n0["sources"][0]["slice"]["kind"] == "time" and n0["sources"][1]["slice"]["kind"] == "char"
assert n0["sources"][0]["locator"]["node_id"] == _r0, "audio ref points at the rendition"
# empty segment: audio ref only, no text_from
n2 = nodes[2]
assert len(n2["sources"]) == 1 and "text_from" not in n2["properties"]
assert n2["properties"]["text"] == ""
print("extension payload checks OK")